[backbone](https://github.com/open-mmlab/mmaction2/blob/main/mmaction/models/backbones/resnet3d_slowfast.py)

In [1]:
%matplotlib inline

import os
import sys

sys.path.append('../../../')

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [2]:
from __future__ import annotations

import copy
import warnings
from collections import OrderedDict
from typing import Optional

import torch
import torch.nn as nn

from computer_vision.slowfast.mmaction.models.backbones.resnet3d_slowfast import ResNet3dPathway, ResNet3dSlowFast

### ResNet3dPathway

In [3]:
slow_pathway={#'type': 'resnet3d', 
    'depth': 50,
  'pretrained': None,
  'lateral': True,
  'conv1_kernel': (1, 7, 7),
  'dilations': (1, 1, 1, 1),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'inflate': (0, 0, 1, 1),
  'norm_eval': False}
fast_pathway={#'type': 'resnet3d',
  'depth': 50,
  'pretrained': None,
  'lateral': False,
  'base_channels': 8,
  'conv1_kernel': (5, 7, 7),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'norm_eval': False}

slow_pathway=ResNet3dPathway(**slow_pathway)
fast_pathway=ResNet3dPathway(**fast_pathway)

In ResNet3dPathway._calculate_lateral_inplanes: depth=50, expansion=4, base_channels=64
stage 0 ----------
	planes=64, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 1 ----------
	planes=256, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 2 ----------
	planes=512, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 3 ----------
	planes=1024, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
conv1_lateral self.inplanes=64, self.channel_ratio=8, (self.inplanes//self.channel_ratio)=8
stage 0 --------------------
planes=64, self.inplanes=256 (i!=self.num_stages-1)=True
lateral_name='layer1_lateral'
stage 1 --------------------
planes=128, self.inplanes=512 (i!=self.num_stages-1)=True
lateral_name='layer2_lateral'
stage 2 --------------------
planes=256, self.inplanes=1024 (i!=self.num_stages-1)=True
lateral_name='layer3_lateral'
stage 3 --------------------
planes=512, self.inplanes=2048 (

In [4]:
x=torch.rand(3,3,10,180,180)
print(f"{x.shape=}, ({x.min().item()}{x.max().item()})")
outs=fast_pathway(x)
print(f'{type(outs)=}, {outs.shape=}, ({outs.min().item()}, {outs.max().item()})')
nn.MSELoss()(outs, torch.rand_like(outs)).backward()

x.shape=torch.Size([3, 3, 10, 180, 180]), (1.7881393432617188e-070.9999998807907104)
type(outs)=<class 'torch.Tensor'>, outs.shape=torch.Size([3, 256, 5, 6, 6]), (0.0, 10.817642211914062)


### ResNet3dSlowFast

In [5]:
kwargs={'type': 'ResNet3dSlowFast',
 'pretrained': None,
 'resample_rate': 8,
 'speed_ratio': 8,
 'channel_ratio': 8,
 'slow_pathway': {'type': 'resnet3d',
  'depth': 50,
  'pretrained': None,
  'lateral': True,
  'conv1_kernel': (1, 7, 7),
  'dilations': (1, 1, 1, 1),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'inflate': (0, 0, 1, 1),
  'norm_eval': False},
 'fast_pathway': {'type': 'resnet3d',
  'depth': 50,
  'pretrained': None,
  'lateral': False,
  'base_channels': 8,
  'conv1_kernel': (5, 7, 7),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'norm_eval': False}}

kwargs.pop('type')
resnet=ResNet3dSlowFast(kwargs)
x=torch.rand(3,3,32,180,180)
print("\nForward pass")
outs=resnet(x, verbose=True)
print(f"outs: {[o.shape for o in outs]}")
loss=0.
for out in outs:
    loss+=nn.MSELoss()(out, torch.rand_like(out))
loss.backward()

In resnet3d_slowfast.ResNet3dSlowFast.__init__ slow_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': True, 'conv1_kernel': (1, 7, 7), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'inflate': (0, 0, 1, 1), 'speed_ratio': 8, 'channel_ratio': 8} 
In resnet3d_slowfast.ResNet3dSlowFast.__init__ fast_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': False, 'base_channels': 8, 'conv1_kernel': (5, 7, 7), 'conv1_stride_t': 1, 'pool1_stride_t': 1} 
In ResNet3dPathway._calculate_lateral_inplanes: depth=50, expansion=4, base_channels=64
stage 0 ----------
	planes=64, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 1 ----------
	planes=256, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 2 ----------
	planes=512, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 3 ----------
	planes=1024, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
conv1_lateral self

In [6]:
output_dirpath='D:/results/ucf101'
fpath=os.path.join(output_dirpath, 'mmaction2-slowfast/demo/slowfast_r50_8xb8-4x16x1-256e_kinetics400_torch.pth')
checkpoint=torch.load(fpath, map_location='cpu', weights_only=False)
resnet.load_state_dict(checkpoint['backbone'])

<All keys matched successfully>